In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

e:\Sentiment_Analysis\env_delta\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
261,This is certainly one of my all time fav episo...,positive
585,Quentin Tarantino's partner in crime Roger Ava...,negative
692,I had two reasons for watching this swashbuckl...,negative
440,I really have problems rating this movie. It i...,negative
244,"If you want a complete waste of time, because ...",negative


In [3]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:32: SyntaxWarning: invalid escape sequence '\s'
<>:32: SyntaxWarning: invalid escape sequence '\s'
C:\Users\chabd\AppData\Local\Temp\ipykernel_14824\3798096103.py:32: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [4]:
df = normalize_text(df)
df.head()

,review,sentiment
261,certainly one time fav episode trek much going...,positive
585,quentin tarantino s partner crime roger avary ...,negative
692,two reason watching swashbuckler aired danish ...,negative
440,really problem rating movie directed brilliant...,negative
244,want complete waste time pulling lint belly bu...,negative


In [5]:
df['sentiment'].value_counts()

sentiment
negative    252
positive    248
Name: count, dtype: int64

In [6]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [7]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
261,certainly one time fav episode trek much going...,1
585,quentin tarantino s partner crime roger avary ...,0
692,two reason watching swashbuckler aired danish ...,0
440,really problem rating movie directed brilliant...,0
244,want complete waste time pulling lint belly bu...,0


In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [14]:
vectorizer = CountVectorizer(max_features=70)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
from dotenv import load_dotenv
import os
import mlflow
import dagshub

load_dotenv()

repo_owner = os.getenv("REPO_OWNER")
repo_name = os.getenv("REPO_NAME")

mlflow.set_tracking_uri(
    f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow"
)

dagshub.init(
    repo_owner=repo_owner,
    repo_name=repo_name,
    mlflow=True
)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


2026-05-12 15:42:13,531 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/chabdullah7/Sentiment_Analysis "HTTP/1.1 200 OK"


Initialized MLflow to track repo "chabdullah7/Sentiment_Analysis"

2026-05-12 15:42:13,531 - INFO - Initialized MLflow to track repo "chabdullah7/Sentiment_Analysis"


Repository chabdullah7/Sentiment_Analysis initialized!

2026-05-12 15:42:13,543 - INFO - Repository chabdullah7/Sentiment_Analysis initialized!


<Experiment: artifact_location='mlflow-artifacts:/44d8d59f68c542b084538daf12c6cdde', creation_time=1778581724601, experiment_id='0', last_update_time=1778581724601, lifecycle_stage='active', name='Logistic Regression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [17]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 70)
        mlflow.log_param("test_size", 0.2)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000) 

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-05-12 15:42:24,407 - INFO - Starting MLflow run...
2026-05-12 15:42:24,929 - INFO - Logging preprocessing parameters...
2026-05-12 15:42:25,895 - INFO - Initializing Logistic Regression model...
2026-05-12 15:42:25,895 - INFO - Fitting the model...
2026-05-12 15:42:25,920 - INFO - Model training complete.
2026-05-12 15:42:25,920 - INFO - Logging model parameters...
2026-05-12 15:42:26,232 - INFO - Making predictions...
2026-05-12 15:42:26,232 - INFO - Calculating evaluation metrics...
2026-05-12 15:42:26,245 - INFO - Logging evaluation metrics...
2026-05-12 15:42:27,497 - INFO - Saving and logging the model...
2026/05/12 15:42:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 15:42:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run learned-hen-838 at: https://dagshub.com/chabdullah7/Sentiment_Analysis.mlflow/#/experiments/0/runs/5ade87e1a00341748a265c10a54aa4f6
🧪 View experiment at: https://dagshub.com/chabdullah7/Sentiment_Analysis.mlflow/#/experiments/0
